In [2]:
import pandas as pd
from tqdm import tqdm

In [7]:
df = pd.read_csv("SharedResponses.csv", nrows=1000)

In [8]:
df.columns

Index(['ResponseID', 'ExtendedSessionID', 'UserID', 'ScenarioOrder',
       'Intervention', 'PedPed', 'Barrier', 'CrossingSignal', 'AttributeLevel',
       'ScenarioTypeStrict', 'ScenarioType', 'DefaultChoice',
       'NonDefaultChoice', 'DefaultChoiceIsOmission', 'NumberOfCharacters',
       'DiffNumberOFCharacters', 'Saved', 'Template', 'DescriptionShown',
       'LeftHand', 'UserCountry3', 'Man', 'Woman', 'Pregnant', 'Stroller',
       'OldMan', 'OldWoman', 'Boy', 'Girl', 'Homeless', 'LargeWoman',
       'LargeMan', 'Criminal', 'MaleExecutive', 'FemaleExecutive',
       'FemaleAthlete', 'MaleAthlete', 'FemaleDoctor', 'MaleDoctor', 'Dog',
       'Cat'],
      dtype='object')

In [2]:
import csv
import sys
from tqdm import tqdm

def count_csv_rows(filepath, encoding='utf-8'):
    """
    Counts the total number of rows in a CSV file efficiently, excluding the header.
    """
    with open(filepath, 'r', encoding=encoding, errors='ignore') as f:
        # The sum(1 for line in f) is a memory-efficient way to count lines.
        # We subtract 1 for the header row.
        return sum(1 for line in f) - 1

def filter_large_csv_with_progress(large_csv_path, small_csv_path, output_csv_path, response_id_column='ResponseID'):
    """
    Filters a large CSV file based on ResponseIDs from a smaller CSV file,
    displaying a progress bar with ETA.

    This function is optimized for memory by reading the large CSV file row by row
    and using a set for fast lookups of the desired ResponseIDs.

    Args:
        large_csv_path (str): The file path for the large master CSV.
        small_csv_path (str): The file path for the smaller CSV containing the ResponseIDs.
        output_csv_path (str): The file path where the filtered output will be saved.
        response_id_column (str): The name of the column containing the ResponseID.
    """
    print("Starting the filtering process...")

    # Step 1: Read all ResponseIDs from the smaller file into a set for efficient lookup.
    try:
        print(f"Loading ResponseIDs from '{small_csv_path}'...")
        with open(small_csv_path, mode='r', encoding='utf-8') as small_file:
            reader = csv.DictReader(small_file)
            # Use a set comprehension for a concise and efficient way to build the set.
            response_ids_to_keep = {row[response_id_column] for row in reader}
    except FileNotFoundError:
        print(f"Error: The file '{small_csv_path}' was not found.")
        return
    except KeyError:
        print(f"Error: Column '{response_id_column}' not found in '{small_csv_path}'.")
        return

    print(f"Loaded {len(response_ids_to_keep)} unique ResponseIDs to keep.")

    # Step 2: Get the total number of rows for the progress bar.
    try:
        print("Calculating total rows in the large file for progress bar...")
        total_rows = count_csv_rows(large_csv_path)
        print(f"Master file has {total_rows} rows to process.")
    except FileNotFoundError:
        print(f"Error: The file '{large_csv_path}' was not found.")
        return

    # Step 3: Stream the large file, filter, and write matching rows to the output file.
    try:
        with open(large_csv_path, mode='r', encoding='utf-8') as large_file, \
             open(output_csv_path, mode='w', encoding='utf-8', newline='') as output_file:

            reader = csv.DictReader(large_file)

            if reader.fieldnames is None:
                print(f"Error: Could not read headers from '{large_csv_path}'.")
                return
            
            if response_id_column not in reader.fieldnames:
                print(f"Error: Column '{response_id_column}' not found in '{large_csv_path}'.")
                return

            writer = csv.DictWriter(output_file, fieldnames=reader.fieldnames)
            writer.writeheader()

            found_count = 0
            
            # Wrap the reader with tqdm to create the progress bar
            progress_bar = tqdm(reader, total=total_rows, unit=" rows", desc="Filtering Master CSV")
            
            for row in progress_bar:
                if row.get(response_id_column) in response_ids_to_keep:
                    writer.writerow(row)
                    found_count += 1
                    # Update a postfix to show how many have been found
                    progress_bar.set_postfix(found=f"{found_count}")

            print(f"\nFiltering complete. A total of {found_count} matching rows were written to '{output_csv_path}'.")

    except Exception as e:
        print(f"An unexpected error occurred during filtering: {e}")

In [9]:
filter_large_csv_with_progress(
    large_csv_path='SharedResponses.csv',
    small_csv_path='SharedResponsesSurvey.csv',
    output_csv_path='filtered_responses.csv',
    response_id_column='ResponseID'
)

Starting the filtering process...
Loading ResponseIDs from 'SharedResponsesSurvey.csv'...
Loaded 5836521 unique ResponseIDs to keep.
Calculating total rows in the large file for progress bar...
Master file has 70332355 rows to process.


Filtering Master CSV: 100%|██████████| 70332355/70332355 [1:05:50<00:00, 17804.73 rows/s, found=11286141] 



Filtering complete. A total of 11286141 matching rows were written to 'filtered_responses.csv'.


In [3]:
# Read the CSV file
df_to_sort = pd.read_csv('filtered_responses.csv')



In [4]:
df_to_sort.head()

,ResponseID,ExtendedSessionID,UserID,ScenarioOrder,Intervention,PedPed,Barrier,CrossingSignal,AttributeLevel,ScenarioTypeStrict,...,LargeMan,Criminal,MaleExecutive,FemaleExecutive,FemaleAthlete,MaleAthlete,FemaleDoctor,MaleDoctor,Dog,Cat
0,2222bRQqBTZ6dLnPH,32757157_6999801415950060.0,6.999801e+15,7,0,0,0,1,Fit,Fitness,...,0,0,0,0,1,2,0,0,0,0
1,22244vvSZfn4J9Zop,1525185249_1436495773909467.0,1.436496e+15,11,0,0,1,0,Low,Social Status,...,0,0,0,0,0,0,0,0,0,0
2,2227h9GkrNbwhBD6t,-1033736141_5392791780749771.0,5.392792e+15,11,0,0,1,0,Male,Gender,...,1,0,0,0,0,0,0,0,0,0
3,222BRvhQN9ZR9LW5h,-972264246_4277296232223713.0,4.277296e+15,12,0,0,1,0,Pets,Species,...,0,0,0,0,0,0,0,0,4,1
4,222Bih22xMQR5brhF,-841718081_3084184331213722.0,3.084184e+15,11,0,0,0,2,Pets,Species,...,0,0,0,0,0,0,0,0,2,2
